# Propensity Score Model Evaluation

Compare two feature schemes for predicting ICI receipt:
1. **Clinical only**: AGE + GENDER + CANCER_TYPE + LINE
2. **Clinical + Embeddings**: AGE + GENDER + CANCER_TYPE + LINE + EMBEDDINGS

Both use elastic-net CV logistic regression with 5-fold held-out scoring.

Evaluation levels:
- Pan-cancer (all patients)
- Within cancer type
- Within line category

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=ConvergenceWarning)

from biomarker_common import DATA_PATH, SURV_PATH, load_note_embeddings
from embed_surv_utils import generate_survival_embedding_df

MATCHING = '1to1'
BUFFER = 30  # day buffer for note embeddings
SEED = 1234

## 1. Load data

In [ ]:
# Matched cohort
COHORT_PATH = os.path.join(DATA_PATH, 'biomarker_analysis/matched_cohorts/')
cohort_df = pd.read_csv(os.path.join(COHORT_PATH, f'matched_cohort_{MATCHING}.csv'))
cohort_df['treatment_start_date'] = pd.to_datetime(cohort_df['treatment_start_date'])
print(f"Cohort: {cohort_df['PX_on_ICI'].sum():.0f} ICI + {(~cohort_df['PX_on_ICI'].astype(bool)).sum()} controls = {len(cohort_df)}")

# Demographics
surv_df = pd.read_csv(os.path.join(SURV_PATH, 'death_met_surv_df.csv'),
                       usecols=['DFCI_MRN', 'GENDER', 'AGE_AT_TREATMENTSTART'])
cohort_df = cohort_df.merge(surv_df.drop_duplicates('DFCI_MRN'), on='DFCI_MRN', how='left')

# Cancer type dummies
cancer_type_df = pd.read_csv(
    os.path.join(DATA_PATH, 'clinical_and_genomic_features/cancer_type_df.csv'))
cancer_type_cols = [c for c in cancer_type_df.columns if c.startswith('CANCER_TYPE_')]
cohort_df = cohort_df.merge(cancer_type_df[['DFCI_MRN'] + cancer_type_cols], on='DFCI_MRN', how='left')

# Line dummies
cohort_df = pd.get_dummies(cohort_df, columns=['line_category'], prefix='LINE', drop_first=True, dtype=int)
line_cols = [c for c in cohort_df.columns if c.startswith('LINE_')]

print(f"Cancer types: {len(cancer_type_cols)}, Line dummies: {len(line_cols)}")
cohort_df.head()

In [ ]:
# Generate embeddings for the chosen buffer
notes_meta, embeddings_data = load_note_embeddings()

note_types = ['Clinician', 'Imaging', 'Pathology']
pool_fx = {nt: 'time_decay_mean' for nt in note_types}

notes_meta_sub = (
    notes_meta[notes_meta['DFCI_MRN'].isin(cohort_df['DFCI_MRN'])]
    .merge(cohort_df[['DFCI_MRN', 'treatment_start_date']].drop_duplicates('DFCI_MRN'),
           on='DFCI_MRN', how='left')
    .assign(NOTE_TIME_REL=lambda df: (
        pd.to_datetime(df['NOTE_DATETIME']) - df['treatment_start_date']).dt.days)
)

embedding_vals = generate_survival_embedding_df(
    notes_meta=notes_meta_sub, survival_df=None, embedding_array=embeddings_data,
    note_types=note_types, note_timing_col='NOTE_TIME_REL',
    max_note_window=-BUFFER, pool_fx=pool_fx, decay_param=0.01, continuous_window=False)

embedding_cols = [c for c in embedding_vals.columns if c != 'DFCI_MRN']
cohort_df = cohort_df.merge(embedding_vals.dropna(), on='DFCI_MRN')
print(f"Patients with embeddings: {len(cohort_df)}, embedding dims: {len(embedding_cols)}")

## 2. Define feature schemes and train

In [ ]:
clinical_cols = ['AGE_AT_TREATMENTSTART', 'GENDER'] + cancer_type_cols + line_cols

SCHEMES = {
    'clinical_only': clinical_cols,
    'clinical_plus_embeddings': clinical_cols + embedding_cols,
}

for name, cols in SCHEMES.items():
    print(f"{name}: {len(cols)} features")

In [ ]:
def train_propensity_cv(df, feature_cols, label, n_splits=5, seed=SEED):
    """Elastic-net CV LR with held-out propensity scores."""
    X = df[feature_cols].values
    y = df['PX_on_ICI'].astype(int).values
    mrns = df['DFCI_MRN'].values
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out_mrns, out_probs, out_true = [], [], []
    fold_info = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        scaler = StandardScaler().fit(X[train_idx])
        X_train_s = scaler.transform(X[train_idx])
        X_test_s = scaler.transform(X[test_idx])
        
        clf = LogisticRegressionCV(
            penalty='elasticnet', solver='saga',
            Cs=8, l1_ratios=[0.1, 0.5, 0.9],
            cv=3, scoring='roc_auc',
            max_iter=2000, tol=1e-3,
            random_state=seed, n_jobs=4,
        )
        clf.fit(X_train_s, y[train_idx])
        
        probs = clf.predict_proba(X_test_s)[:, 1]
        out_mrns.extend(mrns[test_idx])
        out_probs.extend(probs)
        out_true.extend(y[test_idx])
        
        n_nz = np.sum(clf.coef_ != 0)
        fold_info.append(f"fold {fold}: C={clf.C_[0]:.4f}, l1={clf.l1_ratio_[0]:.2f}, feats={n_nz}/{clf.coef_.size}")
    
    auc = roc_auc_score(out_true, out_probs)
    print(f"  [{label}] AUC={auc:.4f}")
    for fi in fold_info:
        print(f"    {fi}")
    
    return pd.DataFrame({'DFCI_MRN': out_mrns, 'ps': out_probs, 'ICI': out_true})

In [ ]:
results = {}
for scheme_name, feat_cols in SCHEMES.items():
    valid = cohort_df.dropna(subset=feat_cols)
    print(f"\n=== {scheme_name} ({len(valid)} patients, {len(feat_cols)} features) ===")
    results[scheme_name] = train_propensity_cv(valid, feat_cols, scheme_name)

## 3. Merge metadata for stratified evaluation

In [ ]:
# Re-load cohort with cancer_type and line_category as labels (not dummies)
cohort_labels = pd.read_csv(os.path.join(COHORT_PATH, f'matched_cohort_{MATCHING}.csv'),
                             usecols=['DFCI_MRN', 'cancer_type', 'line_category'])

for scheme_name in results:
    results[scheme_name] = results[scheme_name].merge(cohort_labels, on='DFCI_MRN', how='left')

results['clinical_only'].head()

## 4. Evaluation functions

In [ ]:
def eval_auc(df, group_col=None, min_per_class=20):
    """Compute AUC + Brier score overall or within groups.
    Requires at least min_per_class in each class to compute AUC."""
    if group_col is None:
        auc = roc_auc_score(df['ICI'], df['ps'])
        brier = brier_score_loss(df['ICI'], df['ps'])
        return pd.DataFrame([{'group': 'overall', 'n': len(df),
                              'n_ICI': df['ICI'].sum(), 'n_ctrl': (1-df['ICI']).sum(),
                              'AUC': auc, 'Brier': brier}])
    
    rows = []
    for grp, gdf in df.groupby(group_col):
        n_pos, n_neg = gdf['ICI'].sum(), (1 - gdf['ICI']).sum()
        if n_pos < min_per_class or n_neg < min_per_class:
            rows.append({'group': grp, 'n': len(gdf), 'n_ICI': n_pos,
                         'n_ctrl': n_neg, 'AUC': np.nan, 'Brier': np.nan})
            continue
        auc = roc_auc_score(gdf['ICI'], gdf['ps'])
        brier = brier_score_loss(gdf['ICI'], gdf['ps'])
        rows.append({'group': grp, 'n': len(gdf), 'n_ICI': n_pos,
                     'n_ctrl': n_neg, 'AUC': auc, 'Brier': brier})
    return pd.DataFrame(rows).sort_values('n', ascending=False)

## 5. Pan-cancer evaluation

In [ ]:
pan_cancer = []
for scheme_name, df in results.items():
    row = eval_auc(df)
    row['scheme'] = scheme_name
    pan_cancer.append(row)
pan_cancer = pd.concat(pan_cancer, ignore_index=True)
print("Pan-cancer AUC:")
pan_cancer[['scheme', 'n', 'n_ICI', 'n_ctrl', 'AUC', 'Brier']]

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(6, 5))
for scheme_name, df in results.items():
    fpr, tpr, _ = roc_curve(df['ICI'], df['ps'])
    auc = roc_auc_score(df['ICI'], df['ps'])
    ax.plot(fpr, tpr, label=f'{scheme_name} (AUC={auc:.3f})')
ax.plot([0,1], [0,1], '--', color='gray')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Pan-cancer ROC: ICI receipt prediction')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Within cancer type

In [ ]:
for scheme_name, df in results.items():
    print(f"\n=== {scheme_name} ===")
    display(eval_auc(df, 'cancer_type'))

In [ ]:
# Side-by-side AUC comparison by cancer type
ct_aucs = []
for scheme_name, df in results.items():
    ea = eval_auc(df, 'cancer_type')
    ea['scheme'] = scheme_name
    ct_aucs.append(ea)
ct_aucs = pd.concat(ct_aucs).dropna(subset=['AUC'])

fig, ax = plt.subplots(figsize=(10, 5))
ct_pivot = ct_aucs.pivot(index='group', columns='scheme', values='AUC').sort_values(
    'clinical_plus_embeddings', ascending=True)
ct_pivot.plot.barh(ax=ax)
ax.set_xlabel('AUC'); ax.set_ylabel('')
ax.set_title('AUC by cancer type')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.legend(title='Scheme')
plt.tight_layout()
plt.show()

## 7. Within line category

In [ ]:
for scheme_name, df in results.items():
    print(f"\n=== {scheme_name} ===")
    display(eval_auc(df, 'line_category'))

In [ ]:
# Side-by-side AUC by line
line_aucs = []
for scheme_name, df in results.items():
    ea = eval_auc(df, 'line_category')
    ea['scheme'] = scheme_name
    line_aucs.append(ea)
line_aucs = pd.concat(line_aucs).dropna(subset=['AUC'])

fig, ax = plt.subplots(figsize=(7, 4))
line_pivot = line_aucs.pivot(index='group', columns='scheme', values='AUC')
line_pivot.plot.barh(ax=ax)
ax.set_xlabel('AUC'); ax.set_ylabel('')
ax.set_title('AUC by line category')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.legend(title='Scheme')
plt.tight_layout()
plt.show()

## 8. PS distribution diagnostics

In [ ]:
fig, axes = plt.subplots(1, len(SCHEMES), figsize=(6 * len(SCHEMES), 4))
if len(SCHEMES) == 1:
    axes = [axes]

for ax, (scheme_name, df) in zip(axes, results.items()):
    for label, grp in df.groupby('ICI'):
        tag = 'ICI' if label == 1 else 'Control'
        ax.hist(grp['ps'], bins=50, alpha=0.5, label=tag, density=True)
    ax.set_title(f'{scheme_name}')
    ax.set_xlabel('Propensity Score')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('PS distributions by treatment arm', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# PS calibration: mean predicted vs actual ICI rate by decile
fig, axes = plt.subplots(1, len(SCHEMES), figsize=(6 * len(SCHEMES), 4))
if len(SCHEMES) == 1:
    axes = [axes]

for ax, (scheme_name, df) in zip(axes, results.items()):
    df = df.copy()
    df['decile'] = pd.qcut(df['ps'], 10, labels=False, duplicates='drop')
    cal = df.groupby('decile').agg(mean_ps=('ps', 'mean'), actual_rate=('ICI', 'mean')).reset_index()
    ax.plot(cal['mean_ps'], cal['actual_rate'], 'o-', label='model')
    ax.plot([0, 1], [0, 1], '--', color='gray', label='perfect')
    ax.set_xlabel('Mean predicted P(ICI)')
    ax.set_ylabel('Actual ICI rate')
    ax.set_title(f'{scheme_name} calibration')
    ax.legend()

plt.tight_layout()
plt.show()

## 9. Summary table

In [ ]:
summary_rows = []

for scheme_name, df in results.items():
    # Pan-cancer
    summary_rows.append({
        'scheme': scheme_name, 'level': 'pan-cancer', 'group': 'all',
        'n': len(df), 'AUC': roc_auc_score(df['ICI'], df['ps']),
        'Brier': brier_score_loss(df['ICI'], df['ps']),
    })
    # By cancer type
    for grp, gdf in df.groupby('cancer_type'):
        if gdf['ICI'].sum() >= 20 and (1 - gdf['ICI']).sum() >= 20:
            summary_rows.append({
                'scheme': scheme_name, 'level': 'cancer_type', 'group': grp,
                'n': len(gdf), 'AUC': roc_auc_score(gdf['ICI'], gdf['ps']),
                'Brier': brier_score_loss(gdf['ICI'], gdf['ps']),
            })
    # By line
    for grp, gdf in df.groupby('line_category'):
        if gdf['ICI'].sum() >= 20 and (1 - gdf['ICI']).sum() >= 20:
            summary_rows.append({
                'scheme': scheme_name, 'level': 'line', 'group': grp,
                'n': len(gdf), 'AUC': roc_auc_score(gdf['ICI'], gdf['ps']),
                'Brier': brier_score_loss(gdf['ICI'], gdf['ps']),
            })

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.round({'AUC': 4, 'Brier': 4})
display(summary_df)

# Save
OUT_DIR = os.path.join(DATA_PATH, f'treatment_prediction/matched_{MATCHING}/ps_evaluation/')
os.makedirs(OUT_DIR, exist_ok=True)
summary_df.to_csv(os.path.join(OUT_DIR, 'ps_evaluation_summary.csv'), index=False)
print(f"Saved to {OUT_DIR}")

In [ ]:
# Save held-out scores
for scheme_name, df in results.items():
    df.to_csv(os.path.join(OUT_DIR, f'held_out_scores_{scheme_name}.csv'), index=False)
    print(f"Saved {scheme_name}: {len(df)} patients")